In [4]:
# This code is adapted from code I found online.  I went back to look for it, so that I could appropriately
# credit its author.  So far, I haven't found it again; Once find it, I'll edit this file to give credit where 
# it's due.

In [5]:
class Applicant:
    def __init__(self,name,program_rankings):
        self.name = name
        self.program_rankings = program_rankings
        self.matched_program  = None
        self.is_matched       = False
      
class Program:
    def __init__(self,name,capacity,applicant_rankings):
        self.name = name
        self.capacity = capacity
        self.applicant_rankings = applicant_rankings
        self.matched_applicants = []

def match(applicants,programs,verbose):

    any_change = True
    
    while any_change:
        any_change = False
        for applicant in applicants:
            if applicant.is_matched:
                continue
            this_appl_change = False    
            for program_name in applicant.program_rankings:
                program = programs[program_name]
                if (applicant.name in program.applicant_rankings) and (len(program.matched_applicants) < program.capacity):
                    program.matched_applicants.append(applicant)
                    applicant.matched_program = program
                    applicant.is_matched      = True 
                    if verbose: print("\f",applicant.name,"added to",program.name)
                    this_appl_change = True
                    break
             
                for matched_applicant in program.matched_applicants:
                    if program.applicant_rankings.index(applicant.name) < program.applicant_rankings.index(matched_applicant.name):
                        program.matched_applicants.remove(matched_applicant)
                        program.matched_applicants.append(applicant)
                        applicant.matched_program         = program
                        applicant.is_matched              = True
                        matched_applicant.matched_program = None
                        matched_applicant.is_matched      = False
                        if verbose: print("\f",applicant.name,"added to",program.name,"replacing ",matched_applicant.name)
                        this_appl_change = True
                        break
                    
                if this_appl_change:
                    any_change = True
                    break
                

In [6]:
import itertools

def print_matches(applicants):
    print("\nMatches:")
    for applicant in applicants:
        if applicant.is_matched:
            print(f" {applicant.name} matched to {applicant.matched_program.name}")
        else:
            print(f" {applicant.name} not matched")

def test_all_applicant_orders():
    applicant_data = [
        ("A", ['PB', 'PA', 'PC', 'PD']), 
        ("B", ['PB', 'PA', 'PC', 'PD']),
        ("C", ['PD', 'PB', 'PA', 'PC']),
        ("D", ['PD', 'PB', 'PA', 'PC']),
        ("E", ['PA', 'PD', 'PB', 'PC']),   
    ]

    program_data = {
        "PA": ("PA", 1, ['C', 'B', 'E', 'D', 'A']),
        "PB": ("PB", 1, ['E', 'C', 'B', 'D', 'A']),
        "PC": ("PC", 1, ['C', 'B', 'D', 'A', 'E']),
        "PD": ("PD", 1, ['D', 'B', 'E', 'A', 'C']),
    }

    all_orders = list(itertools.permutations(applicant_data))

    reference_result = None
    mismatches = 0
    for index, order in enumerate(all_orders):
        
        current_applicants = [Applicant(name, prefs.copy()) for name, prefs in order]
        current_programs = {
            key: Program(name, cap, prefs.copy()) 
            for key, (name, cap, prefs) in program_data.items()
        }

        match(current_applicants, current_programs, verbose=False)

        run_result = []
        for applicant in current_applicants:
            if applicant.is_matched:
                run_result.append((applicant.name, applicant.matched_program.name))
            else:
                run_result.append((applicant.name, None))
        
        run_result.sort() 

        if reference_result is None:
            reference_result = run_result
            print("Reference outcome from the first permutation:")
            sorted_applicants = sorted(current_applicants, key=lambda x: x.name)
            print_matches(sorted_applicants)
        else:
            if run_result != reference_result:
                print(f"Mismatch found at permutation {index}!")
                mismatches += 1


    if mismatches == 0:
        print("All permutations resulted in the exact same matching.")
    else:
        print(f"Ordering of applicants resulted in {mismatches} mismatches.")

test_all_applicant_orders()

Reference outcome from the first permutation:

Matches:
 A not matched
 B matched to PC
 C matched to PA
 D matched to PD
 E matched to PB
All permutations resulted in the exact same matching.


# Define Question
Let there be a group/set of applicants 'A' and a set of programs 'P'.\
We know that the matching algorithm is stable if there does not exist an application 'a' and a program 'p' who are not matched together but both prefer eachother than their current assignments.

We are to prove that regardless of applicant sequence that each applicant will be paired with their best valid partner. Since each applicant will get their best valid partner then the outcome will be a stable, unique, and order-independant match.

# proof by contradiction
we claim that the output matching 'M' is stable.

For the contradiction we assume that there exists a pair ('a', 'p') that prefer eachother but are matched with other applicants/programs.

Since applicant 'a' prefers 'p', they will always propose to that program, which only has two outcomes.

1. They are accepted by 'p'\
Which means that pair ('a', 'p') exists at the final matching output. This contradicts our assumption of a "wrong" pairing.

2. They are rejected by 'p'\
This means that at the time of applicant 'a' proposing to 'p', program 'p' was already holding applicant 'a*' that was ranked higher/more preferable.\
This contradicts with our assumption that ('a', 'p') prefer eachother since 'p' prefers 'a*'

# Conclusion
There does not exist a pair of applicant and program that prefer eachother but or not matched to eachother. No applicant is rejected by their end matching program. Thus making a stable matching output which implies that the suquence of applicants does not affect the matching.